# Prepare individual Raiders posts for HMApp

**Output:** one CSV containing only `Post Text`, with one complete readable post per row.
Select a category, retrieve posts using its seeds, inspect the result, and export.
Run the cells in order from the project root, using the project Python environment.

## 1. Hate categories, anchors and retrieval seeds

The five categories and anchor variants below come from **Wu et al. (2026), Appendix A,
printed page 35 (PDF page 34)**: [local paper](Documentation/a-computational-approach.pdf).
Vocabulary is written in lowercase for readability; matching ignores case and the saved
post text retains its capitalization. Published spellings such as `jewish lighting` are retained.

**Anchors** are the known expressions used by HMApp for semantic comparison.
**Seeds** select which posts enter the CSV. As agreed, each category initially uses its
published anchor variants as seeds; this is our retrieval configuration, not a claim that
these are the paper's exact experimental seed lists. You can edit seeds without changing anchors.

In [ ]:
anchor_terms_by_hate_category = {
    "antisemitism": {
        "kike": ["kike"],
        "blood libel": ["blood libel"],
        "the goyim know": ["the goyim know"],
        "great replacement": ["great replacement"],
        "holocaust/gas chambers": ["holocaust", "gas chambers"],
        "holohoax": ["holohoax"],
        "jew coup": ["jew coup"],
        "jew down": ["jew down"],
        "jewish feature": ["jewish feature"],
        "jewish figure": ["jewish figure"],
        "jewish lighting": ["jewish lighting"],
        "jewish power": ["jewish power"],
        "kosher tax": ["kosher tax"],
        "laughing jew": ["laughing jew"],
        "nazi symbols": ["nazi symbols"],
        "protocols of the elders of zion": ["protocols of the elders of zion"],
        "qanon": ["qanon"],
        "smirking merchant": ["smirking merchant"],
        "zionism is racism": ["zionism is racism"],
        "zionist occupied government": ["zionist occupied government"],
    },
    "anti_black": {
        "nigger": ["nigger", "niger", "nig", "nigor", "nigra", "nigar", "niggur",
                   "nigga", "niggah", "niggar", "nigguh", "niggress", "nigette",
                   "negro", "neger"],
        "jiggaboo": ["jiggaboo", "jiggerboo", "niggerboo", "jiggabo", "jigarooni", "jijjiboo"],
        "cotton picker": ["cotton picker"],
        "moon cricket": ["moon cricket"],
        "darky": ["darky", "darkey", "darkie"],
        "pickaninny": ["pickaninny"],
        "porch monkey": ["porch monkey"],
        "groid": ["groid"],
        "jungle bunny": ["jungle bunny"],
        "sambo": ["sambo"],
        "uncle tom": ["uncle tom"],
        "golliwog": ["golliwog"],
        "spear chucker": ["spear chucker"],
        "alligator bait": ["alligator bait", "’gator bait"],
        "nig-nog": ["nig-nog", "nog", "nignog"],
        "coon": ["coon", "cooney"],
        "buckwheat": ["buckwheat"],
        "spook": ["spook"],
        "ape": ["ape"],
        "jim crow": ["jim crow", "crow"],
        "spade": ["spade"],
        "blackie": ["blackie"],
    },
    "islamophobia": {
        "muzat": ["muzat"],
        "muzzies": ["muzzies"],
        "pislam": ["pislam"],
        "mohammedan": ["mohammedan"],
        "mudslime": ["mudslime"],
        "turk": ["turk"],
    },
    "anti_asian": {
        "slant eye": ["slant eye"],
        "chinaman": ["chinaman"],
        "yellow asian": ["yellow asian"],
        "banana": ["banana"],
        "twinkie": ["twinkie"],
        "ching chong": ["ching chong"],
        "chink": ["chink", "chinky"],
        "chonky": ["chonky"],
        "coolie": ["coolie"],
        "curry-muncher": ["curry-muncher"],
        "zipper head": ["zipper head"],
        "pajeet": ["pajeet"],
        "yellow invader": ["yellow invader"],
        "rice nigger": ["rice nigger"],
        "mongoloid": ["mongoloid"],
        "gook": ["gook"],
    },
    "anti_lgbtq": {
        "faggot": ["faggot", "fagot", "faggotry", "faggoty", "fag", "faggy", "fagged"],
        "homo": ["homo"],
        "tranny": ["tranny"],
        "swishy": ["swishy", "swisher"],
        "dyke": ["dyke", "bulldyke", "bulldagger"],
        "twink": ["twink"],
        "chaser": ["chaser"],
        "gay lord": ["gay lord", "gaylord"],
        "fairy": ["fairy"],
        "daisy": ["daisy", "pansy", "tulip"],
        "cornhole": ["cornhole", "cornholer"],
        "flamer": ["flamer", "flaming"],
        "alphabet people": ["alphabet people"],
        "fruit": ["fruit", "fruity"],
        "fag / hag": ["fag", "hag"],
        "friend of dorothy": ["friend of dorothy"],
        "nancy boy": ["nancy boy"],
        "lipstick lesbian": ["lipstick lesbian"],
        "batting for the other team": ["batting for the other team"],
        "punker": ["punker"],
        "capon": ["capon"],
        "bean queen": ["bean queen"],
        "curry queen": ["curry queen"],
        "harry hoofter": ["harry hoofter"],
    },
}

# Independent lists: editing seeds does not change the anchor dictionaries.
seed_words_by_hate_category = {
    hate_category: list(dict.fromkeys(
        anchor_variant
        for anchor_variants in category_anchor_terms.values()
        for anchor_variant in anchor_variants
    ))
    for hate_category, category_anchor_terms in anchor_terms_by_hate_category.items()
}

## 2. Select the category and input settings

Edit `selected_hate_category` and optionally `selected_seed_words` below.
The initial settings retrieve the **first 1,000 matching antisemitism posts** in file order.
Set `maximum_matching_posts = None` to scan the whole extracted file and keep all matches.
This is an ordered subset, not random sampling.

After changing settings or seeds, rerun the remaining cells in order.
Existing output files are never overwritten: choose another output filename for another run.

In [ ]:
import json
import re
from collections import Counter
from html.parser import HTMLParser
from pathlib import Path

import pandas as pd
from IPython.display import display

selected_hate_category = "antisemitism"
source_dataset_path = Path("data/raiders/raiders_first_1GB.ndjson")
maximum_matching_posts = 1000  # None keeps all matches in the input file.
output_csv_path = Path(f"data/raiders/raiders_{selected_hate_category}.csv")

if selected_hate_category not in anchor_terms_by_hate_category:
    raise ValueError(
        f"Unknown hate category: {selected_hate_category!r}. "
        f"Choose from {list(anchor_terms_by_hate_category)}."
    )
selected_anchor_terms = anchor_terms_by_hate_category[selected_hate_category]
selected_seed_words = seed_words_by_hate_category[selected_hate_category].copy()
# To choose your own seeds, replace selected_seed_words with a list of strings here.

if not source_dataset_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {source_dataset_path.resolve()}")
if maximum_matching_posts is not None and (
    type(maximum_matching_posts) is not int or maximum_matching_posts <= 0
):
    raise ValueError("maximum_matching_posts must be a positive integer or None.")
if not selected_seed_words or any(
    not isinstance(seed_word, str) or not seed_word.strip()
    for seed_word in selected_seed_words
):
    raise ValueError("Provide at least one nonempty seed word or phrase.")
if output_csv_path.exists():
    raise FileExistsError(f"Output already exists; choose another filename: {output_csv_path}")
if not output_csv_path.parent.is_dir():
    raise FileNotFoundError(f"Output directory does not exist: {output_csv_path.parent}")

print("Category:", selected_hate_category)
print("Retrieval seeds:", selected_seed_words)
print("Maximum matching posts:", maximum_matching_posts)
print("Output CSV:", output_csv_path)

## 3. Convert post HTML to readable text and prepare matching

The source stores each post's HTML in `com`. The parser removes markup, decodes entities
such as `&amp;`, and preserves line breaks, capitalization and punctuation. `<wbr>` does not
introduce a space into a word or URL. Visible text is retained, including quote labels;
no quote links or reply relationships are extracted.

Phrase matching ignores case, accepts whitespace between phrase words, and avoids matches
inside larger words. The text exported for HMApp is not lowercased, tokenized or stripped
of stopwords: those operations already belong to your HMApp pipeline.

In [ ]:
class ReadablePostHTMLParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.text_fragments = []

    def handle_starttag(self, tag_name, attributes):
        if tag_name in {"br", "p", "div", "li"}:
            self.text_fragments.append("\n")

    def handle_endtag(self, tag_name):
        if tag_name in {"p", "div", "li"}:
            self.text_fragments.append("\n")

    def handle_data(self, text_fragment):
        self.text_fragments.append(text_fragment)


def extract_readable_post_text(post_html):
    post_html_parser = ReadablePostHTMLParser()
    post_html_parser.feed(post_html)
    post_html_parser.close()
    readable_post_text = "".join(post_html_parser.text_fragments)
    readable_post_text = re.sub(r"[^\S\n]+", " ", readable_post_text)
    return "\n".join(
        text_line.strip() for text_line in readable_post_text.splitlines()
    ).strip()


def compile_phrase_pattern(phrase):
    return re.compile(
        r"(?<!\w)"
        + r"\s+".join(re.escape(phrase_word) for phrase_word in phrase.split())
        + r"(?!\w)",
        flags=re.IGNORECASE,
    )


seed_patterns = {
    seed_word: compile_phrase_pattern(seed_word)
    for seed_word in selected_seed_words
}
anchor_patterns = {
    anchor_name: [compile_phrase_pattern(variant) for variant in anchor_variants]
    for anchor_name, anchor_variants in selected_anchor_terms.items()
}

## 4. Retrieve individual matching posts

Each source line contains a `posts` list. We iterate over that list and evaluate every post
independently. A post is retained once if **its own readable text** matches any selected seed.
Matching multiple seeds does not add extra copies of a post. Identical text already present
in different source posts is retained; there is no deduplication.

Reading stops at the configured number of matches or the end of the file. Empty-text posts
are counted and skipped. Malformed records cause an explicit error with their source line.

In [ ]:
matched_post_texts = []
seed_match_counts = Counter()
examined_post_count = 0
empty_text_post_count = 0
first_matched_post_html = None
matching_post_limit_reached = False

with source_dataset_path.open(encoding="utf-8") as source_dataset_file:
    for source_line_number, source_line in enumerate(source_dataset_file, start=1):
        try:
            source_record = json.loads(source_line)
        except json.JSONDecodeError as parsing_error:
            raise ValueError(f"Invalid JSON on source line {source_line_number}.") from parsing_error
        if not isinstance(source_record, dict) or not isinstance(source_record.get("posts"), list):
            raise ValueError(f"Expected a posts list on source line {source_line_number}.")

        for source_post in source_record["posts"]:
            examined_post_count += 1
            if examined_post_count % 100_000 == 0:
                print(f"Examined {examined_post_count:,} posts; retained {len(matched_post_texts):,}.")
            if not isinstance(source_post, dict):
                raise ValueError(f"Expected a post object on source line {source_line_number}.")
            post_html = source_post.get("com")
            if post_html is None:
                empty_text_post_count += 1
                continue
            if not isinstance(post_html, str):
                raise ValueError(f"Expected com to be text on source line {source_line_number}.")

            readable_post_text = extract_readable_post_text(post_html)
            if not readable_post_text:
                empty_text_post_count += 1
                continue
            matching_seed_words = [
                seed_word
                for seed_word, seed_pattern in seed_patterns.items()
                if seed_pattern.search(readable_post_text)
            ]
            if matching_seed_words:
                matched_post_texts.append(readable_post_text)
                seed_match_counts.update(matching_seed_words)
                if first_matched_post_html is None:
                    first_matched_post_html = post_html
                if maximum_matching_posts is not None and len(matched_post_texts) >= maximum_matching_posts:
                    matching_post_limit_reached = True
                    break
        if matching_post_limit_reached:
            break

if not matched_post_texts:
    raise ValueError("No posts matched the selected seeds. Inspect the seeds and source dataset.")

hmapp_input_posts = pd.DataFrame({"Post Text": matched_post_texts})
print("Stopped because:", "matching-post limit reached" if matching_post_limit_reached else "end of file")
print(f"Posts examined: {examined_post_count:,}")
print(f"Posts skipped for empty text: {empty_text_post_count:,}")
print(f"Posts selected: {len(hmapp_input_posts):,}")

## 5. Inspect the posts and anchor coverage

Inspect example texts and how the HTML was converted. The tables count how many retained
posts contain each seed or anchor. One post can count towards several terms; these counts
are not hate labels or evaluation scores. An anchor is counted once per post even when
several of its variants occur. Missing anchors remain visible with a zero count.

Your current HMApp constructs anchor embeddings from the input posts. If none of the
selected category's anchors occurs, stop here rather than exporting an input without
anchor examples. This check does not modify which posts were selected.

In [ ]:
with pd.option_context("display.max_colwidth", 250):
    display(hmapp_input_posts.head(5))

print("First matching post - original HTML:")
print(first_matched_post_html)
print("First matching post - readable text:")
print(hmapp_input_posts.iloc[0]["Post Text"])

seed_coverage = pd.DataFrame([
    {"seed": seed_word, "matching_posts": seed_match_counts[seed_word]}
    for seed_word in seed_patterns
]).sort_values("matching_posts", ascending=False)
display(seed_coverage)

anchor_coverage = pd.DataFrame([
    {
        "anchor": anchor_name,
        "matching_posts": sum(
            any(anchor_pattern.search(post_text) for anchor_pattern in anchor_variant_patterns)
            for post_text in hmapp_input_posts["Post Text"]
        ),
    }
    for anchor_name, anchor_variant_patterns in anchor_patterns.items()
]).sort_values("matching_posts", ascending=False)
display(anchor_coverage)

represented_anchor_count = int(anchor_coverage["matching_posts"].gt(0).sum())
print(f"Anchors represented: {represented_anchor_count} / {len(selected_anchor_terms)}")
if represented_anchor_count == 0:
    raise ValueError("No selected-category anchor examples in the retrieved posts. Inspect the seeds and post limit.")

## 6. Export the HMApp input CSV

This cell writes only `Post Text`, with no pandas index column, and reloads the CSV to
verify that all texts are preserved exactly. CSV quoting handles commas, quotation marks
and embedded line breaks. `mode="x"` refuses to overwrite an existing file.

The CSV contains complete matching posts only. Changing the runner's input path and
selecting the corresponding scoring anchors will be handled separately, as agreed.

In [ ]:
if list(hmapp_input_posts.columns) != ["Post Text"] or hmapp_input_posts.empty:
    raise ValueError("Expected a nonempty table containing only the Post Text column.")
if not hmapp_input_posts["Post Text"].map(
    lambda post_text: isinstance(post_text, str) and bool(post_text.strip())
).all():
    raise ValueError("Every exported post must contain nonempty text.")

hmapp_input_posts.to_csv(output_csv_path, index=False, encoding="utf-8", mode="x")
reloaded_hmapp_input_posts = pd.read_csv(
    output_csv_path, keep_default_na=False, dtype={"Post Text": str}, encoding="utf-8"
)
if not reloaded_hmapp_input_posts.equals(hmapp_input_posts):
    raise ValueError("The exported CSV does not reproduce the selected posts exactly.")

print(f"Saved {len(reloaded_hmapp_input_posts):,} posts to {output_csv_path.resolve()}")
display(reloaded_hmapp_input_posts.head())